<span STYLE="font-size:150%"> 
    Segment microCT scans
</span>

Docker image: gnasello/slicer-env:2023-07-06 \
Latest update: 10 March 2023

- load image stack in Slicer
- segment mineralized tissue
- compute segmented statistics (volumes)

# Load libraries

In [ ]:
import pyslicer as ps
import slicer
from pathlib import Path
import pandas as pd
import pyvista as pv

# Volume input

## Load `.nrrd` file

Write the path of the `.nrrd` file and load it to Slicer

In [ ]:
# this cell is tagged 'parameters'
volume_file = 'microCT_volume/microCT_volume.nrrd'
segment_file = 'segmented_volumes/Bone.seg.nrrd'
output_dir_path = 'segmented_volumes'

In [ ]:
volume_file = 'microCT_volume/microCT_volume_preview.nrrd'
segment_file = 'segmented_volumes/Bone.seg.nrrd'
output_dir_path = 'segmented_volumes'

In [ ]:
file_path = Path(volume_file)

# Remove image numbering _0000, _0001 ...
filename_output = file_path.stem[:-4]

In [ ]:
masterVolumeNode = slicer.util.loadNodeFromFile(file_path.resolve())

Print spacing

In [ ]:
## mm
masterVolumeNode.GetSpacing()

Make ```segmented_volumes``` folder

In [ ]:
output_directory = Path(output_dir_path)

output_directory.mkdir(parents=True, exist_ok=True)

In [ ]:
# Monitor Memory in Slicer
import psutil, os
print(psutil.Process(os.getpid()).memory_info().rss / (1024**3), "GB used")

# Create segmentationNode

## Create segmentation-related nodes

Create segmentation node

In [ ]:
file_path = Path(segment_file)

segmentationNode = slicer.util.loadSegmentation(file_path.resolve(), properties={'name':"Segmentation"})

Create temporary segment editor to get access to effects

In [ ]:
segmentEditorWidget, segmentEditorNode = ps.segmentation.segmentEditorWidget(segmentationNode = segmentationNode, 
                                                                             masterVolumeNode = masterVolumeNode)

# Operation on segments

## Manual fix of the segmentation

Sometimes it might be necessary to remove speckles at the image boundaries. If so, use the `scissor` tool in the `Segment Editor` before proceeding with the rest of the script. 

## Remove small islands

REMOVE_SMALL_ISLANDS operation from the [SegmentEditorIslandsEffect](https://github.com/Slicer/Slicer/blob/294ef47edbac2ccb194d5ee982a493696795cdc0/Modules/Loadable/Segmentations/EditorEffects/Python/SegmentEditorIslandsEffect.py#L402)

In [ ]:
segment_name = 'Segment_1'
minimum_size = 20 #number of voxels

In [ ]:
ps.segmentation.remove_small_islands(minimum_size, 
                                     segment_name, 
                                     segmentEditorNode,
                                     segmentEditorWidget)

## Smoothing thresholded segment

In [ ]:
segment_name = 'Segment_1'
gaussiaSD_mm = 0.001 #units are in millimiters

In [ ]:
ps.segmentation.gaussian_smoothing(gaussiaSD_mm, 
                                   segment_name, 
                                   segmentEditorNode, 
                                   segmentEditorWidget)

# Export model as .stl file

## Convert all segments to model nodes

Get closed surface representation of the segment, from [slicer scripting repository](https://slicer.readthedocs.io/en/latest/developer_guide/script_repository.html#export-nodes-warped-by-transform-sequence)

In [ ]:
segmentationNode.CreateClosedSurfaceRepresentation()

In [ ]:
shNode = slicer.mrmlScene.GetSubjectHierarchyNode()
exportFolderItemId = shNode.CreateFolderItem(shNode.GetSceneItemID(), "Segments")
slicer.modules.segmentations.logic().ExportAllSegmentsToModels(segmentationNode, exportFolderItemId)

## Decimate model node from segmentation node

Get `VTK Polydata` from model

In [ ]:
nodename = 'Segment_1'
modelNode = slicer.util.getNode(nodename)

Use pyvista to get faces and vertices of the `VTK PolyData` object

In [ ]:
import pyvista as pv

In [ ]:
model_pv = pv.wrap(modelNode.GetPolyData())
vertices = model_pv.points
faces = model_pv.regular_faces

In 3D Slicer, you can clear the scene using Python scripting by calling the Clear method on the application's MRML scene.

In [ ]:
#slicer.mrmlScene.Clear(0)

Import pyvista object to pymeshlab, which performs more efficient decimation than VTK

Check how to [import mesh from arrays in meshlab](https://pymeshlab.readthedocs.io/en/0.1.9/tutorials/import_mesh_from_arrays.html)

In [ ]:
import pymeshlab 
# create a new Mesh with the two arrays
model_ml = pymeshlab.Mesh(vertices, faces)

# create a new MeshSet
mesh_set = pymeshlab.MeshSet()

# add the mesh to the MeshSet
mesh_set.add_mesh(model_ml, "mesh")

print('input mesh has', model_ml.vertex_number(), 'vertex and', model_ml.face_number(), 'faces')

Decimate mesh with `pymeshlab`, from [stackoverflow](https://stackoverflow.com/questions/75169329/how-to-use-pymeshlab-to-reduce-number-of-faces-to-a-guaranteed-specific-number)

In [ ]:
decimation_factor = 0.2

#Target number of vertex
TARGET = round(model_ml.vertex_number() * decimation_factor)

#Reduce to TARGET. Sometimes will fall into TARGET-1
mesh_set.meshing_decimation_quadric_edge_collapse(targetfacenum=TARGET, preservenormal=True)

model_ml = mesh_set.current_mesh()
print('output mesh has', model_ml.vertex_number(), 'vertex and', model_ml.face_number(), 'faces')

Load decimated model to Slicer

In [ ]:
vertices = model_ml.vertex_matrix()
faces = model_ml.face_matrix()
model_decimated = pv.PolyData.from_regular_faces(vertices, faces)

decimatedNode = slicer.modules.models.logic().AddModel(model_decimated)

## Export decimated model from segmentation node

Export pymeshlab object to .stl file

In [ ]:
filename_output = 'Bone.stl'

slicer.util.exportNode(decimatedNode, output_directory / filename_output)